In [1]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")
include("../src/flexOPT.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

  Activating 

devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]

project at `~/Documents/Github/flexOPT`



→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


Main.GeoPoints

In [12]:
using .commonBatchs, UnPack, Symbolics

In [13]:

function integralBsplineTaylorKernels1DWithWindow1D(BsplineOrder,WBsplineOrder,μᶜ,μ,ν,L,Δ::Float64,l_n_variable,l_n_field)
    
    # Δ should be strictly Float64

    # this computes the analytical value of the 1D integral between B-spline fns and weighted Taylor kernels
    # \int dx Bspline Y_μᶜ Y_μ  K_{lᶜ-nᶜ}(y-y_μᶜ) K_{l-n}(y-y_μ)

    # unlike the previous integralBsplineTaylorKernels1D, it computes for a specific ν
    # Cˡη;μ are computed for a specific geometry, so even though the boundary condition reduces
    # the number of available points, each Taylor expansion for K_{l-n}(y-y_μ) should be Ok
    
    # however, the 'forgotten' μ (due to the whole) should be treated carefully 
    # (which I have not yet implemented here): I think Y_ignoredμ should be added to the Y_availableneighbouringμ

    # or maybe the 'forgotten' μ is anyways not available (and thus very probably not continuous)
    # so we just let this be forgotten 

        

    #@show μᶜ,μ,ν,L,Δ,l_n_variable,l_n_field

    kernelValue=0.0
   
    maximumOrder = maximum((BsplineOrder,WBsplineOrder,0)) # even if we only use the indicator functions (classical FD), we will call it for modμ (which we do not use for classical FD)
    params=@strdict maximumOrder numberNodes = L

    #output,_=@produce_or_load(BsplineTimesPolynomialsIntegrated,params,datadir("BsplineInt");filename = config -> savename("Bspline",params))
    
    output=myProduceOrLoad(BsplineTimesPolynomialsIntegrated,params,"BsplineInt","Bspline")
    #@show output
    nodeIndices,nodesSymbolic,b_deriv,integral_b,Δx,extFns,x,modμ =output["BsplineIntegraters"]

    @show "coucou "
    @show modμ
    if BsplineOrder=== -1
        # this is for an indicator function
        if l_n_variable === 0 && l_n_field === 0
            kernelValue=1.0
        else
            kernelValue=0.0
        end

    else
        
        
        # here we make a function Y_μ' Y_μ K_μ' K_μ (details ommitted)
        # note that ν is somewhere middle or at extremeties and 'ν+' expression is ommitted 

        Y_μᶜ = zeros(Num,L)
        Y_μ = zeros(Num,L)

        if WBsplineOrder === -1

            if μᶜ === ν
                Y_μᶜ = ones(Num,L) 
            end

            if μ === ν
                Y_μ = ones(Num,L)
            end

        else
            Y_μᶜ=b_deriv[:,μᶜ,1,WBsplineOrder+1]
            Y_μ =b_deriv[:,μ ,1,WBsplineOrder+1]
        end

        # modμ[3,:,ι+1] is the symbolic expression of the centre to compute Taylor kernels, which can be staggered!!!

        K_μᶜ=(x-Δx*modμ[3,μᶜ,WBsplineOrder+1])^l_n_variable
        K_μ =(x-Δx*modμ[3,μ,WBsplineOrder+1])^l_n_field

        # the convoluted function of all above
        F = mySimplify.(Y_μᶜ .* Y_μ .* K_μᶜ .* K_μ)


        # the target kernel integral

        targetKernel = integral_b[ν,BsplineOrder+1]

        dictionaryForSubstitute = Dict()
    
        for i in 0:1:BsplineOrder
            F = integrateTaylorPolynomials.(F,x) # integrate already for the 1st partial of W
            
            # mathematically I need to understand why but F cannot be disturbed by supplementary complexities due to constants 
            # that are arbitrarily put during the integral
            
            #F .-= substitute(F[ν],Dict(x=>nodesSymbolic[ν]))

            # F should be continuous, in general but since all the integral by parts
            # is done piecewisely we might not need this
            #=
            for iSegment in 2:1:L # this should be sequential
                lastValue = substitute(F[iSegment-1],Dict(x=>nodesSymbolic[iSegment]))
                startValue = substitute(F[iSegment],Dict(x=>nodesSymbolic[iSegment]))
                shiftValue = lastValue - startValue
                F[iSegment]=F[iSegment]+shiftValue
            end
            =#

            F .= mySimplify(F)

            for iSegment in nodeIndices
                dictionaryForSubstitute[extFns[1,iSegment,i+1]]=substitute(F[iSegment],Dict(x=>nodesSymbolic[iSegment]))
                dictionaryForSubstitute[extFns[2,iSegment,i+1]]=substitute(F[iSegment],Dict(x=>nodesSymbolic[iSegment+1]))
            end
        end

        #@show dictionaryForSubstitute,targetKernel
        

        kernelValue = substitute(targetKernel,dictionaryForSubstitute)  
        
        kernelValue = substitute(kernelValue,Dict(Δx=>Δ))/(BigInt(factorial(l_n_field))*BigInt(factorial(l_n_variable)))
        kernelValue = Num2Float64(kernelValue)
        
        #a= (Δ^(l_n_variable+l_n_field+1)-(-Δ)^(l_n_variable+l_n_field+1))/((l_n_variable+l_n_field+2)*(l_n_variable+l_n_field+1)*factorial(BigInt(l_n_variable))*factorial(BigInt(l_n_field)))
        #@show a
    end
    return kernelValue,modμ
    
end

integralBsplineTaylorKernels1DWithWindow1D (generic function with 1 method)

In [14]:


function BsplineTimesPolynomialsIntegrated(params::Dict)

    @unpack maximumOrder, numberNodes = params
    @variables x Δx ξ
   
    extFns = Symbolics.variables(:extFns,1:2,1:numberNodes,1:maximumOrder+1)
    # maximum order of B-spline
    
    maximumOrder = maximumOrder + 1

    

    ∂x = Differential(x)
    
    # input parameters
    
  
    
    # the left and right indices
    
    νₗ = 1 # we cannot touch this anymore
    νᵣ = numberNodes # like in the middle we know that the expression is ok 

    
    # let's start
    
    
    nodeIndices = collect(νₗ:1:νᵣ)
    nodesSymbolic = Δx .* nodeIndices
    append!(nodesSymbolic,nodesSymbolic[end])


    
    # b-splines
    
    b = zeros(Num, numberNodes, numberNodes, maximumOrder)
    bX = zeros(Num,numberNodes, 2, maximumOrder)
    
    # b-splines derivatives
    
    b_deriv = zeros(Num,numberNodes, numberNodes,maximumOrder,maximumOrder)
   
    for ι in 0:1:maximumOrder-1
        
        local neighbour = Int((1 - (-1)^ι) / 2)
        floor = Int((ι - neighbour) / 2)
        ceiling = Int((ι + neighbour) / 2)
        if ι === 0
            for ν in nodeIndices # this will run for all the ν related to nodes
                tmpν = ν - νₗ + 1
            
                b[tmpν, tmpν, ι+1] = 1
            end
        else
    
            for ν in nodeIndices # this will run for all the ν related to nodes
                tmpν = ν - νₗ + 1
                # the denominator for the ι>0
    
                denominator = BigInt(ι) * Δx
    
                # for the upgoing part
    
                if ν - neighbour <= νᵣ && ν - neighbour >= νₗ
                    rightlimit = minimum((numberNodes, tmpν + floor))
                    leftlimit = maximum((1, tmpν - ceiling))
                    for νSegment in leftlimit:1:rightlimit
                        tmpνSegment = νSegment #- νₗ + 1
                        numerator = x - BigInt(ν - ceiling) * Δx
                        b[tmpνSegment, tmpν, ι+1] += mySimplify(numerator / denominator * b[tmpνSegment, tmpν-neighbour, ι])
                    end
                end
    
                #for the downgoing part
    
                if ν - neighbour + 1 <= νᵣ && ν - neighbour + 1 >= νₗ
                    rightlimit = minimum((numberNodes, tmpν + floor + 1))
                    leftlimit = maximum((1, tmpν - ceiling + 1))
                    for νSegment in leftlimit:1:rightlimit
                        tmpνSegment = νSegment #- νₗ + 1
                        numerator = BigInt(ν + floor + 1) * Δx - x
                        b[tmpνSegment, tmpν, ι+1] += mySimplify(numerator / denominator * b[tmpνSegment, tmpν-neighbour+1, ι])
                    end
    
                end
    
            end
        end
    
        # unfortunately the νₗ and νᵣ related functions should recover the 'lost' amplitudes due to the trunctation 
        # we need to re-build based on the residual
    
        for ν in nodeIndices
            tmpν = ν - νₗ + 1
            
            #b[:,tmpν,ι+1] .= 0
            if ν===νₗ  # this loop runs only for the extremeties!!!!!
                # the nodes involved at most are: νₗ, νₗ + 1, ⋯, νₗ + ceiling 
                bX[1:ceiling+1,1,ι+1] .=1
                for νSegment in 1:1:ceiling+1
                    for νFunction in 2:1:numberNodes
                        bX[νSegment,1,ι+1] -= b[νSegment,νFunction,ι+1]
                    end
                end
            elseif ν===νᵣ
                bX[numberNodes-ceiling-1:numberNodes,2,ι+1] .=1
                for νSegment in numberNodes-ceiling-1:1:numberNodes
                    for νFunction in 1:1:numberNodes-1
                        bX[νSegment,2,ι+1] -= b[νSegment,νFunction,ι+1]
                    end
                end
            end
    
        end
    

        b[:,1,ι+1]=bX[:,1,ι+1]
        b[:,end,ι+1]=bX[:,end,ι+1]
        
          
        
        # computing the derivatives 
    
        for i in 0:1:maximumOrder-1 
            if i === 0
                b_deriv[:,:,i+1,ι+1] = b[:,:,ι+1]
            else
                b_deriv[:,:,i+1,ι+1] = mySimplify.(∂x.(b_deriv[:,:,i,ι+1]))
            
            end
    
        end
    
    end
        
    
    # re-calculated centre
    
    modμ=zeros(Num,3,numberNodes,maximumOrder) # min, max, mid for each b-splines this can be a multiple of Δx / 2 

    for ι in 0:1:maximumOrder-1
        for μ in nodeIndices
            leftLimit = nodesSymbolic[findfirst(x->x!==0,b[:,μ,ι+1])] /Δx
            rightLimit = nodesSymbolic[findlast(x->x!==0,b[:,μ,ι+1])+1] /Δx
            midNode = mySimplify((leftLimit+rightLimit)/2)
            modμ[1,μ,ι+1]=leftLimit 
            modμ[2,μ,ι+1]=rightLimit
            modμ[3,μ,ι+1]=midNode
        end
    end
    

    integral_b= zeros(Num,numberNodes,maximumOrder)

    #gvec = (@variables (g(x))[1:maximumOrder])[1]
    
    

    for ι in 0:1:maximumOrder-1
        for i in 0:1:maximumOrder-1
            for ν in nodeIndices # this will run for all the ν related to nodes
                tmpν = ν - νₗ + 1
                for νSegment in nodeIndices
                    tmpνSegment = νSegment - νₗ + 1
                    
                    integral_b[tmpν,ι+1] -= (-1)^(i)*extFns[1,tmpνSegment,i+1]*substitute(b_deriv[tmpνSegment,tmpν,i+1,ι+1],Dict(x=>nodesSymbolic[tmpνSegment]))
    
                    integral_b[tmpν,ι+1] += (-1)^(i)*extFns[2,tmpνSegment,i+1]*substitute(b_deriv[tmpνSegment,tmpν,i+1,ι+1],Dict(x=>nodesSymbolic[tmpνSegment+1]))
                    
                end
            end
        end
    end
    
    
    integral_b=mySimplify.(integral_b)
    
    BsplineIntegraters=(nodeIndices=nodeIndices,nodesSymbolic=nodesSymbolic,b_deriv=b_deriv,integral_b=integral_b,Δx=Δx,extFns=extFns,x=x,modμ=modμ)
    return @strdict(BsplineIntegraters)

end
    

BsplineTimesPolynomialsIntegrated (generic function with 1 method)

In [15]:
BsplineOrder=-1
WBsplineOrder=-1
μᶜ=1
μ=1
ν=1
L=3
Δ=1.0
l_n_variable=0
l_n_field=0

0

In [16]:
integralBsplineTaylorKernels1DWithWindow1D(BsplineOrder,WBsplineOrder,μᶜ,μ,ν,L,Δ,l_n_variable,l_n_field)

"coucou " = "coucou "
modμ = Num[1 1 1; 3 3 3; 2.0 2.0 2.0;;;]


(1.0, Num[1 1 1; 3 3 3; 2.0 2.0 2.0;;;])